# Siamese BiLSTM: training on the case 2s pairs

Character-level bi-encoder (`model.py`): char embedding 32 → BiLSTM 1 × 128 → masked mean or max pooling → head on `[|u−v|; u⊙v]`. Protocol (`training.py`): BCE, Adam 1e-3, at most 20 epochs, patience 3, best epoch = most val non-identical matches found among epochs that reject ≥ 98% of val case 1, ties → lower val loss. One seed (0) per pooling, as in the paper. About 4 min per epoch on an M1 Pro.

Writes `checkpoints/siamese/` (git-ignored) and `predictions/siamese_{mean,max}/` (history + val/test probabilities). A run whose checkpoint exists is skipped.

In [ ]:
import os
import sys
sys.path.append("..")   # repository root -> lets us import helpers/ and the model packages

import numpy as np
import torch
from torch.utils.data import DataLoader

from helpers.pairs import load_pairs
from helpers.paths import probs_path
from siamese.model import build_vocab, PairDataset, make_collate
from siamese.training import (train_run, load_run, rebuild_model, predict, broken_run_problems,
                              checkpoint_path, history_path, run_name, CHECKPOINT_DIR)

DATASET = "overture_case2s"
SEED = 0
POOLINGS = ["mean", "max"]
BATCH_SIZE, LR, MAX_EPOCHS, PATIENCE = 64, 1e-3, 20, 3
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## 1. Data, vocabulary (train split only), loaders

In [ ]:
train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
vocab = build_vocab(train)
print(len(train), len(val), len(vocab))

train_loader = DataLoader(PairDataset(train, train_label), batch_size=BATCH_SIZE, shuffle=True, collate_fn=make_collate(vocab))
val_loader = DataLoader(PairDataset(val, val_label), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(vocab))

## 2. Train and save (skipped if the checkpoint exists)

In [ ]:
for pooling in POOLINGS:
    if not os.path.exists(checkpoint_path(SEED, pooling)):
        history, best, best_state = train_run(SEED, pooling, vocab, train_loader, val_loader, val, val_label,
                                              lr=LR, max_epochs=MAX_EPOCHS, patience=PATIENCE)
        os.makedirs(os.path.dirname(history_path(SEED, pooling)), exist_ok=True)
        history.to_csv(history_path(SEED, pooling), index=False)
        torch.save({"state_dict": best_state, "vocab": vocab,
                    "config": {"emb_dim": 32, "hidden": 128, "pooling": pooling, "max_len": 85},
                    "seed": SEED, "best": best}, checkpoint_path(SEED, pooling))
    history, ckpt = load_run(SEED, pooling)
    print(f"{pooling}: best epoch {ckpt['best']}")
    print(f"{pooling}: problems {broken_run_problems(history)}")

## 3. Val / test probabilities → `predictions/`

In [ ]:
for pooling in POOLINGS:
    _, ckpt = load_run(SEED, pooling)
    model = rebuild_model(ckpt)
    for split in ["val", "test"]:
        df, labels = load_pairs(DATASET, split)
        loader = DataLoader(PairDataset(df, labels), batch_size=BATCH_SIZE, shuffle=False, collate_fn=make_collate(ckpt["vocab"]))
        np.save(probs_path(run_name(pooling), split, SEED), predict(model, loader).numpy())